In [3]:
# Cell 1: Imports and config (robust to missing PennyLane)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED = 42
np.random.seed(SEED)

# Try to import PennyLane; if unavailable try to pip-install it; otherwise
# fall back to a classical MLP-based placeholder that mimics a QNN interface.
USE_PENNYLANE = False
try:
    import pennylane as qml
    from pennylane import numpy as pnp
    USE_PENNYLANE = True
except Exception:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pennylane'], cwd='/tmp')
        import pennylane as qml
        from pennylane import numpy as pnp
        USE_PENNYLANE = True
    except Exception as e:
        print('Warning: PennyLane not available (installation failed or not permitted). Using classical fallback. Error:', e)
        from sklearn.neural_network import MLPClassifier
        pnp = np
        qml = None
        class QNNFallback:
            """Simple sklearn MLP wrapper to stand in for a QNN in demos."""
            def __init__(self, hidden_layer_sizes=(10,), max_iter=200, random_state=SEED):
                self.clf = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, max_iter=max_iter, random_state=random_state)
            def fit(self, X, y):
                self.clf.fit(X, y)
                return self
            def predict(self, X):
                return self.clf.predict(X)
        # Export fallback symbol expected by downstream cells
        QNNModel = QNNFallback

# When PennyLane is present, user code should construct quantum nodes/devices as usual.
# When absent, use QNNModel as a drop-in classical alternative.


In [4]:
# Cell 2: Tiny dataset (Iris-like binary or toy)
X = np.random.randn(200, 2)
y = ((X[:, 0] * X[:, 1]) > 0).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)


In [5]:
# Cell 3: Device and model
n_qubits = X.shape[1]
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

def ansatz(weights):
    # One variational layer: Rot on each qubit then ring entanglement
    for w, wire in zip(weights, range(n_qubits)):
        qml.Rot(w[0], w[1], w[2], wires=wire)
    # simple entanglement
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])
    if n_qubits > 1:
        qml.CNOT(wires=[n_qubits - 1, 0])

@qml.qnode(dev)
def circuit(x, weights):
    # Angle embedding
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


In [6]:
# Cell 4: Prediction and loss
def predict_proba(Xb, weights, bias):
    # map expval [-1,1] -> probability via sigmoid on affine
    z = pnp.array([circuit(x, weights) for x in Xb])
    logits = z + bias
    return 1.0 / (1.0 + pnp.exp(-logits))

def predict_label(Xb, weights, bias, thr=0.5):
    p = predict_proba(Xb, weights, bias)
    return (p >= thr).astype(int)

def loss_fn(Xb, yb, weights, bias):
    p = predict_proba(Xb, weights, bias)
    eps = 1e-9
    # binary cross-entropy
    return -pnp.mean(yb * pnp.log(p + eps) + (1 - yb) * pnp.log(1 - p + eps))


In [7]:
# Cell 5: Train loop (simple gradient descent)
layers = 1
weights = pnp.random.normal(scale=0.1, size=(n_qubits, 3), requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.1)

epochs = 60
for ep in range(epochs):
    weights, bias, cost = opt.step(lambda w, b: loss_fn(X_tr, y_tr, w, b), weights, bias), None, None
    weights, bias = weights[0], weights[1]
    cost = loss_fn(X_tr, y_tr, weights, bias)
    if (ep+1) % 10 == 0:
        print(f"Epoch {ep+1}: loss={float(cost):.4f}")

y_hat = predict_label(X_te, weights, bias)
print("Test accuracy:", accuracy_score(y_te, np.array(y_hat)))


Epoch 10: loss=0.7503
Epoch 20: loss=0.7351
Epoch 20: loss=0.7351
Epoch 30: loss=0.7255
Epoch 30: loss=0.7255
Epoch 40: loss=0.7192
Epoch 40: loss=0.7192
Epoch 50: loss=0.7152
Epoch 50: loss=0.7152
Epoch 60: loss=0.7126
Test accuracy: 0.5
Epoch 60: loss=0.7126
Test accuracy: 0.5
